# Data Analysis (Part 1)
All Dataframes using Pandas, Polars and PySpark will be named by "df", "dp" and "data", respectively.

In [1]:
# Import libraries
import polars as pl
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pyspark
from pyspark.sql import SparkSession, SQLContext
from pyspark.sql.types import IntegerType
from pyspark.sql import functions as F

In [2]:
# SparkSession
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Local")
    .master("local[*]") # For local mode with all available cores
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "1")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("--- Successful PySpark Configuration ---")
print(f"Execution Mode: {spark.conf.get('spark.master')}")
print(f"Driver Memory: {spark.conf.get('spark.driver.memory')}")
print(f"Executor Memory: {spark.conf.get('spark.executor.memory')}")

26/01/04 12:17:45 WARN Utils: Your hostname, ant resolves to a loopback address: 127.0.1.1; using 192.168.0.33 instead (on interface wlp1s0)
26/01/04 12:17:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/04 12:17:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


--- Successful PySpark Configuration ---
Execution Mode: local[*]
Driver Memory: 2g
Executor Memory: 2g


## DataFrames

In [4]:
# Create a simple DataFrame
df = pd.DataFrame({"Nombre":["Cruz","Antonio"], "ID":[1,2]})

# Show the first rows of the DataFrame
print(df.head())

    Nombre  ID
0     Cruz   1
1  Antonio   2


In [5]:
dp = pl.DataFrame({"Nombre":["Cruz","Antonio"], "ID":[1,2]})
print(dp.head())

shape: (2, 2)
┌─────────┬─────┐
│ Nombre  ┆ ID  │
│ ---     ┆ --- │
│ str     ┆ i64 │
╞═════════╪═════╡
│ Cruz    ┆ 1   │
│ Antonio ┆ 2   │
└─────────┴─────┘


In [6]:
data = spark.createDataFrame([("Cruz", 1), ("Antonio", 2)], ["Nombre", "ID"])
data.show()

+-------+---+
| Nombre| ID|
+-------+---+
|   Cruz|  1|
|Antonio|  2|
+-------+---+



## Read from CSV file

In [6]:
# Create DataFrame from CSV file
df = pd.read_csv('../data/games.csv',sep=',',header=0)
dp = pl.read_csv('../data/games.csv',separator=',',has_header=True)
print(dp.head(3))

shape: (3, 5)
┌───────────────────┬──────────┬─────────────────┬──────────┬───────┐
│ name              ┆ platform ┆ year_of_release ┆ genre    ┆ sales │
│ ---               ┆ ---      ┆ ---             ┆ ---      ┆ ---   │
│ str               ┆ str      ┆ i64             ┆ str      ┆ f64   │
╞═══════════════════╪══════════╪═════════════════╪══════════╪═══════╡
│ Wii Sports        ┆ Wii      ┆ 2006            ┆ Sports   ┆ 41.36 │
│ Super Mario Bros. ┆ NES      ┆ 1985            ┆ Platform ┆ 29.08 │
│ Mario Kart Wii    ┆ Wii      ┆ 2008            ┆ Racing   ┆ 15.68 │
└───────────────────┴──────────┴─────────────────┴──────────┴───────┘


In [7]:
data = spark.read.csv('../data/games.csv', header=True, inferSchema=True, sep=',')
data.show(3)

+-----------------+--------+---------------+--------+-----+
|             name|platform|year_of_release|   genre|sales|
+-----------------+--------+---------------+--------+-----+
|       Wii Sports|     Wii|           2006|  Sports|41.36|
|Super Mario Bros.|     NES|           1985|Platform|29.08|
|   Mario Kart Wii|     Wii|           2008|  Racing|15.68|
+-----------------+--------+---------------+--------+-----+
only showing top 3 rows



In [9]:
# Or
data.head(3)

[Row(name='Wii Sports', platform='Wii', year_of_release=2006, genre='Sports', sales=41.36),
 Row(name='Super Mario Bros.', platform='NES', year_of_release=1985, genre='Platform', sales=29.08),
 Row(name='Mario Kart Wii', platform='Wii', year_of_release=2008, genre='Racing', sales=15.68)]

## Read from Excel file

In [4]:
# Read Excel file (with no header in the file)
df1 = pd.read_excel('../data/users.xlsx', sheet_name='Sheet1', header=None, names=['user_id','name'])

## Save to file

In [ ]:
# Save to csv file
# df.to_csv('../data/pandas_output.csv', index=False)
# dp.write_csv('../data/polars_output.csv')

In [ ]:
# Save to csv file (a folder will be created with one file, coalesce(1))
# data.coalesce(1).write.mode("overwrite").option("header", "true").csv('../data/pyspark_output.csv')

## Display

In [10]:
# DataFrame tail
print(df.tail(3))
print(dp.tail(3))
spark.createDataFrame(df.tail(3)).show()

                          name platform  year_of_release       genre  sales
16712  Haitaka no Psychedelica      PSV           2016.0   Adventure   0.00
16713         Spirits & Spells      GBA           2003.0    Platform   0.01
16714      Winning Post 8 2016      PSV           2016.0  Simulation   0.00
shape: (3, 5)
┌─────────────────────────┬──────────┬─────────────────┬────────────┬───────┐
│ name                    ┆ platform ┆ year_of_release ┆ genre      ┆ sales │
│ ---                     ┆ ---      ┆ ---             ┆ ---        ┆ ---   │
│ str                     ┆ str      ┆ i64             ┆ str        ┆ f64   │
╞═════════════════════════╪══════════╪═════════════════╪════════════╪═══════╡
│ Haitaka no Psychedelica ┆ PSV      ┆ 2016            ┆ Adventure  ┆ 0.0   │
│ Spirits & Spells        ┆ GBA      ┆ 2003            ┆ Platform   ┆ 0.01  │
│ Winning Post 8 2016     ┆ PSV      ┆ 2016            ┆ Simulation ┆ 0.0   │
└─────────────────────────┴──────────┴─────────────────┴──

/home/ant/PySpark-Polars-Pandas/env/lib/python3.8/site-packages/pyspark/sql/pandas/conversion.py:351: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  PyArrow >= 4.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


+--------------------+--------+---------------+----------+-----+
|                name|platform|year_of_release|     genre|sales|
+--------------------+--------+---------------+----------+-----+
|Haitaka no Psyche...|     PSV|         2016.0| Adventure|  0.0|
|    Spirits & Spells|     GBA|         2003.0|  Platform| 0.01|
| Winning Post 8 2016|     PSV|         2016.0|Simulation|  0.0|
+--------------------+--------+---------------+----------+-----+



In [11]:
# DataFrame sample
print(df.sample(3, random_state=0))
print(dp.sample(n=3, seed=0))
data.sample(withReplacement=False, fraction=0.1, seed=0).show(3)

                               name platform  year_of_release   genre  sales
7634   Press Your Luck 2010 Edition       DS           2009.0    Misc   0.18
13771                     Aeon Flux      PS2           2005.0  Action   0.02
3051   Castlevania: Lords of Shadow     X360           2010.0  Action   0.42
shape: (3, 5)
┌─────────────────────────────────┬──────────┬─────────────────┬─────────┬───────┐
│ name                            ┆ platform ┆ year_of_release ┆ genre   ┆ sales │
│ ---                             ┆ ---      ┆ ---             ┆ ---     ┆ ---   │
│ str                             ┆ str      ┆ i64             ┆ str     ┆ f64   │
╞═════════════════════════════════╪══════════╪═════════════════╪═════════╪═══════╡
│ SpongeBob SquarePants: Game Bo… ┆ GBA      ┆ 2004            ┆ Misc    ┆ 0.15  │
│ Castle Shikigami 2              ┆ PS2      ┆ 2004            ┆ Shooter ┆ 0.01  │
│ Mat Hoffman's Pro BMX 2         ┆ XB       ┆ 2002            ┆ Sports  ┆ 0.15  │
└─────────────

## Dataframe sShape and columns types

In [12]:
# DataFrame shape
print(df.shape)
print(dp.shape)  # (rows, columns)
print((data.count(), len(data.columns)))

(16715, 5)
(16715, 5)
(16715, 5)


In [13]:
# DataFrame columns
print(df.columns)
print(dp.columns)
print(data.columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'sales'], dtype='object')
['name', 'platform', 'year_of_release', 'genre', 'sales']
['name', 'platform', 'year_of_release', 'genre', 'sales']


In [14]:
# DataFrame columns types
print(df.dtypes)

name                object
platform            object
year_of_release    float64
genre               object
sales              float64
dtype: object


In [15]:
print(dp.dtypes)
print(data.dtypes)

[String, String, Int64, String, Float64]
[('name', 'string'), ('platform', 'string'), ('year_of_release', 'int'), ('genre', 'string'), ('sales', 'double')]


In [16]:
# Type of the DataFrame
type(df)

pandas.core.frame.DataFrame

In [17]:
type(dp)

polars.dataframe.frame.DataFrame

In [18]:
type(data)

pyspark.sql.dataframe.DataFrame

## Information and statistics

In [19]:
# Show DataFrame Info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16715 entries, 0 to 16714
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16713 non-null  object 
 1   platform         16715 non-null  object 
 2   year_of_release  16446 non-null  float64
 3   genre            16713 non-null  object 
 4   sales            16715 non-null  float64
dtypes: float64(2), object(3)
memory usage: 653.1+ KB


In [20]:
# Show DataFrame Info
print(dp.glimpse())

Rows: 16715
Columns: 5
$ name            <str> 'Wii Sports', 'Super Mario Bros.', 'Mario Kart Wii', 'Wii Sports Resort', 'Pokemon Red/Pokemon Blue', 'Tetris', 'New Super Mario Bros.', 'Wii Play', 'New Super Mario Bros. Wii', 'Duck Hunt'
$ platform        <str> 'Wii', 'NES', 'Wii', 'Wii', 'GB', 'GB', 'DS', 'Wii', 'Wii', 'NES'
$ year_of_release <i64> 2006, 1985, 2008, 2009, 1996, 1989, 2006, 2006, 2009, 1984
$ genre           <str> 'Sports', 'Platform', 'Racing', 'Sports', 'Role-Playing', 'Puzzle', 'Platform', 'Misc', 'Platform', 'Shooter'
$ sales           <f64> 41.36, 29.08, 15.68, 15.61, 11.27, 23.2, 11.28, 13.96, 14.44, 26.93

None


In [21]:
# Show DataFrame schema
data.printSchema()

root
 |-- name: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- year_of_release: integer (nullable = true)
 |-- genre: string (nullable = true)
 |-- sales: double (nullable = true)



In [22]:
# DataFrame statistics
print(df.describe())
print(dp.describe())

       year_of_release         sales
count     16446.000000  16715.000000
mean       2006.484616      0.263377
std           5.877050      0.813604
min        1980.000000      0.000000
25%        2003.000000      0.000000
50%        2007.000000      0.080000
75%        2010.000000      0.240000
max        2016.000000     41.360000
shape: (9, 6)
┌────────────┬──────────────────────────────┬──────────┬─────────────────┬──────────┬──────────┐
│ statistic  ┆ name                         ┆ platform ┆ year_of_release ┆ genre    ┆ sales    │
│ ---        ┆ ---                          ┆ ---      ┆ ---             ┆ ---      ┆ ---      │
│ str        ┆ str                          ┆ str      ┆ f64             ┆ str      ┆ f64      │
╞════════════╪══════════════════════════════╪══════════╪═════════════════╪══════════╪══════════╡
│ count      ┆ 16713                        ┆ 16715    ┆ 16446.0         ┆ 16713    ┆ 16715.0  │
│ null_count ┆ 2                            ┆ 0        ┆ 269.0         

In [23]:
data.describe().show()
# or
data.summary().show()

26/01/04 11:56:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+--------------------+--------+------------------+--------+-------------------+
|summary|                name|platform|   year_of_release|   genre|              sales|
+-------+--------------------+--------+------------------+--------+-------------------+
|  count|               16713|   16715|             16446|   16713|              16715|
|   mean|              1942.0|  2600.0|2006.4846163200777|    NULL|0.26337720610232485|
| stddev|                NULL|     0.0| 5.877049828016608|    NULL| 0.8136035222632941|
|    min|      Beyblade Burst|    2600|              1980|  Action|                0.0|
|    max|¡Shin Chan Flipa ...|    XOne|              2016|Strategy|              41.36|
+-------+--------------------+--------+------------------+--------+-------------------+



+-------+--------------------+--------+------------------+--------+-------------------+
|summary|                name|platform|   year_of_release|   genre|              sales|
+-------+--------------------+--------+------------------+--------+-------------------+
|  count|               16713|   16715|             16446|   16713|              16715|
|   mean|              1942.0|  2600.0|2006.4846163200777|    NULL|0.26337720610232485|
| stddev|                NULL|     0.0| 5.877049828016608|    NULL| 0.8136035222632941|
|    min|      Beyblade Burst|    2600|              1980|  Action|                0.0|
|    25%|              1942.0|  2600.0|              2003|    NULL|                0.0|
|    50%|              1942.0|  2600.0|              2007|    NULL|               0.08|
|    75%|              1942.0|  2600.0|              2010|    NULL|               0.24|
|    max|¡Shin Chan Flipa ...|    XOne|              2016|Strategy|              41.36|
+-------+--------------------+--

## Null values

In [24]:
# Verify null values per column
print(df.isna().sum())
print(dp.null_count())
data.select([pyspark.sql.functions.count(pyspark.sql.functions.when(pyspark.sql.functions.col(c).isNull(), c)).alias(c) for c in data.columns]).show()

name                 2
platform             0
year_of_release    269
genre                2
sales                0
dtype: int64
shape: (1, 5)
┌──────┬──────────┬─────────────────┬───────┬───────┐
│ name ┆ platform ┆ year_of_release ┆ genre ┆ sales │
│ ---  ┆ ---      ┆ ---             ┆ ---   ┆ ---   │
│ u32  ┆ u32      ┆ u32             ┆ u32   ┆ u32   │
╞══════╪══════════╪═════════════════╪═══════╪═══════╡
│ 2    ┆ 0        ┆ 269             ┆ 2     ┆ 0     │
└──────┴──────────┴─────────────────┴───────┴───────┘
+----+--------+---------------+-----+-----+
|name|platform|year_of_release|genre|sales|
+----+--------+---------------+-----+-----+
|   2|       0|            269|    2|    0|
+----+--------+---------------+-----+-----+



In [25]:
# Fill null values
df['name'] = df['name'].fillna('Unknown')

dp = dp.with_columns(
    pl.col("name").fill_null(value="Unknown")
)

data = data.na.fill({"name": "Unknown"})

In [26]:
# Fill null values with strategy
df['sales'] = df['sales'].fillna(df['sales'].mean())

dp = dp.with_columns(
    pl.col("sales").fill_null(strategy="mean") # 'forward', 'backward', 'min', 'max', 'mean'
)

from pyspark.sql import functions as F
data = data.na.fill({'sales': data.agg(F.avg('sales')).collect()[0][0]})

In [27]:
# Drop rows with null values
df = df.dropna()
dp = dp.drop_nulls()
data = data.na.drop()

In [28]:
# Drop rows with null values in specific columns
df = df.dropna(subset=['name', 'platform'])
dp = dp.drop_nulls(subset=['name', 'platform'])
data = data.na.drop(subset=['name', 'platform'])

In [29]:
# Show unique values
# print(df['platform'].unique())
# print(dp['platform'].unique())
# data.select('platform').distinct().show()

In [30]:
# Count unique values
print(df['platform'].nunique())
print(dp['platform'].n_unique())
data.select(pyspark.sql.functions.countDistinct('platform')).show()

31
31
+------------------------+
|count(DISTINCT platform)|
+------------------------+
|                      31|
+------------------------+



In [31]:
# Count values in column
# print(df['platform'].value_counts())
# print(dp['platform'].value_counts())
# data.groupBy('platform').count().show()

In [32]:
# Extract specific columns
df_cut = df[['name','platform']]
dp_cut = dp.select(['name','platform'])

In [33]:
# Extract specific columns
data_cut = data[['name','platform']]
# or
data_cut = data.select(['name','platform'])

In [34]:
# Extract info by column
df_cut = df.loc[:,['name','platform']]
dp_cut = dp.select(['name','platform'])
data_cut = data.select('name','platform').show(3)

+-----------------+--------+
|             name|platform|
+-----------------+--------+
|       Wii Sports|     Wii|
|Super Mario Bros.|     NES|
|   Mario Kart Wii|     Wii|
+-----------------+--------+
only showing top 3 rows



In [35]:
# Extract info by column and row
df_cut = df.loc[0:3,['name','platform']]

dp_cut = dp[0:4, ["name", "platform"]]
# or
dp_cut = dp.select(['name', 'platform']).slice(0, 4)
# or
dp_cut = dp.select(pl.col('name'), pl.col('platform')).head(4)

data_cut = data.select('name', 'platform').limit(4).show()

+-----------------+--------+
|             name|platform|
+-----------------+--------+
|       Wii Sports|     Wii|
|Super Mario Bros.|     NES|
|   Mario Kart Wii|     Wii|
|Wii Sports Resort|     Wii|
+-----------------+--------+



In [36]:
# Extract info in consecutive columns
df_cut = df.loc[0:3,'name':'platform']

import polars.selectors as cs
dp_cut = dp.select(pl.col("^name|platform$")).slice(0, 4)
dp_cut.head(2)

name,platform
str,str
"""Wii Sports""","""Wii"""
"""Super Mario Bros.""","""NES"""


In [37]:
cols = data.columns
idx_start = cols.index("name")
idx_end = cols.index("platform")
target_cols = cols[idx_start : idx_end + 1]
data_cut = data.select(*target_cols).limit(2).show()

+-----------------+--------+
|             name|platform|
+-----------------+--------+
|       Wii Sports|     Wii|
|Super Mario Bros.|     NES|
+-----------------+--------+



In [38]:
# Extract info by column and row
df_cut = df.iloc[0:3,0:2]

dp_cut = dp[0:4, 0:2]

data_cut = data.limit(4).select(data.columns[0:2]).show()

+-----------------+--------+
|             name|platform|
+-----------------+--------+
|       Wii Sports|     Wii|
|Super Mario Bros.|     NES|
|   Mario Kart Wii|     Wii|
|Wii Sports Resort|     Wii|
+-----------------+--------+



In [39]:
# Rename columns
df_r = df.rename(columns={'name':'Game_Name','platform':'Game_Platform'})
dp_r = dp.rename({'name':'Game_Name','platform':'Game_Platform'})
data_r = data.withColumnRenamed('name', 'Game_Name').withColumnRenamed('platform', 'Game_Platform')

In [40]:
# DataFrame columns
print(df_r.columns)
print(dp_r.columns)
print(data_r.columns)

Index(['Game_Name', 'Game_Platform', 'year_of_release', 'genre', 'sales'], dtype='object')
['Game_Name', 'Game_Platform', 'year_of_release', 'genre', 'sales']
['Game_Name', 'Game_Platform', 'year_of_release', 'genre', 'sales']


In [41]:
# Column operation
df['sales2'] = df['sales']*2

dp = dp.with_columns(
    (pl.col('sales') * 2).alias('sales2')
)
dp.head(3)

name,platform,year_of_release,genre,sales,sales2
str,str,i64,str,f64,f64
"""Wii Sports""","""Wii""",2006,"""Sports""",41.36,82.72
"""Super Mario Bros.""","""NES""",1985,"""Platform""",29.08,58.16
"""Mario Kart Wii""","""Wii""",2008,"""Racing""",15.68,31.36


In [42]:
data = data.withColumn('sales2', data['sales']*2)
data.show(3)

+-----------------+--------+---------------+--------+-----+------+
|             name|platform|year_of_release|   genre|sales|sales2|
+-----------------+--------+---------------+--------+-----+------+
|       Wii Sports|     Wii|           2006|  Sports|41.36| 82.72|
|Super Mario Bros.|     NES|           1985|Platform|29.08| 58.16|
|   Mario Kart Wii|     Wii|           2008|  Racing|15.68| 31.36|
+-----------------+--------+---------------+--------+-----+------+
only showing top 3 rows



In [43]:
# Delete row duplicates based on specific columns
df = df.drop_duplicates(subset=['name','platform'])
dp = dp.unique(subset=['name','platform'])
data = data.dropDuplicates(['name','platform'])

In [44]:
# Count duplicates
print(df.duplicated(subset=['name','platform']).sum())
print(dp['name', 'platform'].is_duplicated().sum())
data.groupBy('name', 'platform').count().filter(F.col('count') > 1).count()

0
0


0